In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_5.py ──
"""
Shared utilities for Exercise 5 — GANs and Generative Models.

Infrastructure only: data loading, visualisation helpers, metric computation,
and kailash-ml engine setup. No domain logic or business scenarios.
"""

import asyncio
import pickle
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import matplotlib.pyplot as plt
from matplotlib.figure import Figure

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


if TYPE_CHECKING:
    from kailash_ml import ModelVersion

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
LATENT_DIM = 64
IMG_DIM = 28 * 28
BATCH_SIZE = 128
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
    OUTPUT_DIR = _HERE.parent
except NameError:
    REPO_ROOT = Path.cwd()
    OUTPUT_DIR = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "mnist"


# ════════════════════════════════════════════════════════════════════════
# Environment and device
# ════════════════════════════════════════════════════════════════════════
def init_environment() -> torch.device:
    """Set up environment, seeds, and return the compute device."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ════════════════════════════════════════════════════════════════════════
# Data loading
# ════════════════════════════════════════════════════════════════════════
def load_mnist(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, DataLoader]:
    """Load full MNIST (60K), scale to [-1, 1] for tanh generators.

    Returns:
        X_real: (60000, 1, 28, 28) tensor on device, range [-1, 1]
        y_real: (60000,) long tensor on device
        real_loader: DataLoader with batch_size=128, shuffle, drop_last
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_real = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_real = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))], dtype=torch.long
    )
    X_real = (X_real * 2.0 - 1.0).to(device)
    y_real = y_real.to(device)

    print(
        f"MNIST: {len(X_real)} digits, shape {tuple(X_real.shape[1:])}, "
        f"pixel range [{X_real.min():.2f}, {X_real.max():.2f}]"
    )
    class_dist = ", ".join(f"{c}={int((y_real == c).sum())}" for c in range(10))
    print(f"  class distribution: {class_dist}")

    real_loader = DataLoader(
        TensorDataset(X_real), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    return X_real, y_real, real_loader


# ════════════════════════════════════════════════════════════════════════
# Kailash engine setup
# ════════════════════════════════════════════════════════════════════════
def setup_engines() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None]
):
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry
    """

    async def _setup():
        # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
        # and ModelRegistry use incompatible _kml_model_versions schemas.
        # Route them to separate sqlite files until upstream fixes the conflict.
        db = "sqlite:///mlfp05_gans.db"
        registry_db = "sqlite:///mlfp05_gans_registry.db"
        tracker = await ExperimentTracker.create(store_url=db)
        conn = ConnectionManager(registry_db)
        await conn.initialize()
        registry = ModelRegistry(conn)
        return conn, tracker, "m5_gans", registry

    return asyncio.run(_setup())


async def close_engines(conn: ConnectionManager) -> None:
    """Cleanly shut down the connection manager."""
    await conn.close()


# ════════════════════════════════════════════════════════════════════════
# Generator and Discriminator architectures
# ════════════════════════════════════════════════════════════════════════
class Generator(nn.Module):
    """MLP Generator: z -> 784-d -> (1, 28, 28).

    Uses BatchNorm + LeakyReLU (DCGAN best practices) and Tanh output
    to match the [-1, 1] image scaling.
    """

    def __init__(self, latent_dim: int = LATENT_DIM, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden * 2),
            nn.BatchNorm1d(hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, IMG_DIM),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    """MLP Discriminator: 28x28 -> scalar logit.

    Dropout prevents D from overfitting to real images (memorising
    instead of learning distributional features).
    """

    def __init__(self, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMG_DIM, hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# FID feature extractor
# ════════════════════════════════════════════════════════════════════════
class LeNetFeatureExtractor(nn.Module):
    """CNN feature extractor for FID computation.

    Returns 64-dim feature vectors (analogous to InceptionV3 pool3 layer).
    We use a domain-specific extractor because InceptionV3 expects 299x299 RGB;
    for 28x28 grayscale MNIST a trained LeNet gives more meaningful distances.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


def train_feature_extractor(
    X_real: torch.Tensor,
    y_real: torch.Tensor,
    device: torch.device,
    epochs: int = 5,
) -> LeNetFeatureExtractor:
    """Train the LeNet feature extractor on MNIST for FID computation.

    Returns the trained extractor in eval mode.
    """
    print("\n== Training feature extractor (for FID + mode coverage) ==")
    extractor = LeNetFeatureExtractor().to(device)
    opt = torch.optim.Adam(extractor.parameters(), lr=1e-3)
    X_01 = (X_real + 1.0) / 2.0  # [0, 1] for classifier

    for epoch in range(epochs):
        losses = []
        for xb, yb in DataLoader(
            TensorDataset(X_01, y_real), batch_size=256, shuffle=True
        ):
            loss = F.cross_entropy(extractor(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        with torch.no_grad():
            acc = (extractor(X_01[:10000]).argmax(-1) == y_real[:10000]).float().mean()
        print(f"  epoch {epoch+1}/{epochs}  loss={np.mean(losses):.3f}  acc={acc:.3f}")

    extractor.eval()
    return extractor


# ════════════════════════════════════════════════════════════════════════
# FID computation
# ════════════════════════════════════════════════════════════════════════
def compute_fid(
    extractor: LeNetFeatureExtractor,
    real: torch.Tensor,
    generated: torch.Tensor,
) -> float:
    """Frechet Inception Distance between real and generated images.

    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r @ Sigma_g))

    Lower FID = closer to real distribution = better generator.
    Uses eigendecomposition (no scipy dependency).
    """
    extractor.eval()

    def _extract(images: torch.Tensor) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for i in range(0, len(images), 512):
                feats.append(
                    extractor.extract_features(images[i : i + 512]).cpu().numpy()
                )
        return np.concatenate(feats)

    rf, gf = _extract(real), _extract(generated)
    mu_r, mu_g = rf.mean(0), gf.mean(0)
    sig_r, sig_g = np.cov(rf, rowvar=False), np.cov(gf, rowvar=False)

    diff = mu_r - mu_g
    product = sig_r @ sig_g
    eigvals, eigvecs = np.linalg.eigh(product)
    eigvals = np.maximum(eigvals, 0.0)
    sqrt_prod = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T

    return float(diff @ diff + np.trace(sig_r + sig_g - 2 * sqrt_prod))


# ════════════════════════════════════════════════════════════════════════
# Mode coverage diagnostic
# ════════════════════════════════════════════════════════════════════════
def mode_coverage(
    G: Generator,
    classifier: LeNetFeatureExtractor,
    device: torch.device,
    n: int = 5000,
) -> tuple[int, dict[int, int], float]:
    """Measure mode coverage: how many of the 10 digit classes the generator produces.

    Returns:
        n_classes: number of unique classes generated
        per_class_counts: dict mapping class -> count
        shannon_entropy: diversity measure (max = log2(10) = 3.32 for uniform)
    """
    G.eval()
    classifier.eval()
    with torch.no_grad():
        fake_01 = (G(torch.randn(n, LATENT_DIM, device=device)) + 1) / 2
        preds = classifier(fake_01).argmax(-1).cpu().numpy()
    unique, counts = np.unique(preds, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
    return int(len(unique)), {int(k): int(v) for k, v in zip(unique, counts)}, entropy


# ════════════════════════════════════════════════════════════════════════
# Visualisation helpers
# ════════════════════════════════════════════════════════════════════════
def plot_image_grid(
    images: torch.Tensor,
    nrow: int = 8,
    ncol: int = 8,
    title: str = "Generated Images",
    save_path: str | None = None,
) -> Figure:
    """Plot an 8x8 grid of generated images.

    Args:
        images: (N, 1, 28, 28) tensor in [-1, 1] range
        nrow, ncol: grid dimensions
        title: figure title
        save_path: optional path to save the figure
    """
    n = min(nrow * ncol, len(images))
    fig, axes = plt.subplots(nrow, ncol, figsize=(12, 12))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for i in range(nrow * ncol):
        ax = axes[i // ncol][i % ncol]
        if i < n:
            img = images[i].detach().cpu().squeeze().numpy()
            img = (img + 1) / 2  # [-1,1] -> [0,1]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_latent_interpolation(
    G: Generator,
    device: torch.device,
    n_steps: int = 10,
    n_rows: int = 5,
    title: str = "Latent Space Interpolation",
    save_path: str | None = None,
) -> Figure:
    """Interpolate between pairs of random latent vectors.

    Shows smooth transitions between generated images — evidence that
    the generator has learned a continuous manifold, not memorised digits.
    """
    G.eval()
    fig, axes = plt.subplots(n_rows, n_steps, figsize=(n_steps * 1.2, n_rows * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    with torch.no_grad():
        for row in range(n_rows):
            z1 = torch.randn(1, LATENT_DIM, device=device)
            z2 = torch.randn(1, LATENT_DIM, device=device)
            for col in range(n_steps):
                alpha = col / (n_steps - 1)
                z = (1 - alpha) * z1 + alpha * z2
                img = G(z).squeeze().cpu().numpy()
                img = (img + 1) / 2
                axes[row][col].imshow(img, cmap="gray", vmin=0, vmax=1)
                axes[row][col].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_training_progression(
    G: Generator,
    device: torch.device,
    epoch_snapshots: dict[int, dict],
    title: str = "Training Progression",
    save_path: str | None = None,
) -> Figure:
    """Plot generated images at different training epochs.

    Shows how generation quality improves over training — from random
    noise to recognisable digits.

    Args:
        epoch_snapshots: dict of {epoch: state_dict} captured during training
    """
    n_snapshots = len(epoch_snapshots)
    n_samples = 8
    fig, axes = plt.subplots(
        n_snapshots, n_samples, figsize=(n_samples * 1.4, n_snapshots * 1.6)
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    fixed_z = torch.randn(n_samples, LATENT_DIM, device=device)

    for row, (epoch, state_dict) in enumerate(sorted(epoch_snapshots.items())):
        G.load_state_dict(state_dict)
        G.eval()
        with torch.no_grad():
            imgs = G(fixed_z)
        for col in range(n_samples):
            ax = axes[row][col] if n_snapshots > 1 else axes[col]
            img = imgs[col].squeeze().cpu().numpy()
            img = (img + 1) / 2
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"Epoch {epoch}", fontsize=10, rotation=0, labelpad=50)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_loss_curves(
    g_losses: list[float],
    d_losses: list[float],
    title: str = "Training Dynamics",
    g_label: str = "Generator",
    d_label: str = "Discriminator",
    save_path: str | None = None,
) -> Figure:
    """Plot G vs D loss curves across epochs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    epochs = range(1, len(g_losses) + 1)

    # Individual losses
    ax1.plot(epochs, g_losses, "b-", linewidth=2, label=g_label)
    ax1.plot(epochs, d_losses, "r-", linewidth=2, label=d_label)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("G vs D Loss", fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # G/D ratio — healthy GAN has ratio near 1
    ratio = [g / (d + 1e-8) for g, d in zip(g_losses, d_losses)]
    ax2.plot(epochs, ratio, "g-", linewidth=2)
    ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Balanced (1.0)")
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("G/D Loss Ratio", fontsize=12)
    ax2.set_title("Training Balance", fontsize=13)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


# ════════════════════════════════════════════════════════════════════════
# Model registration helper
# ════════════════════════════════════════════════════════════════════════
def register_generator(
    registry: ModelRegistry | None,
    name: str,
    model: Generator,
    fid: float,
    coverage: int,
    entropy: float,
) -> "ModelVersion | None":
    """Register a trained generator in ModelRegistry with quality metrics."""
    if registry is None:
        print(f"  ModelRegistry not available — skipping {name}")
        return None

    async def _register():
        ver = await registry.register_model(
            name=f"m5_{name}",
            artifact=pickle.dumps(model.state_dict()),
            metrics=[
                MetricSpec(name="fid_score", value=fid),
                MetricSpec(name="mode_coverage", value=float(coverage)),
                MetricSpec(name="class_entropy", value=entropy),
            ],
        )
        print(
            f"  Registered {name}: v={ver.version}, FID={fid:.2f}, "
            f"coverage={coverage}/10"
        )
        return ver

    return asyncio.run(_register())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 5.1: Vanilla GAN (The Minimax Game)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - The adversarial minimax game: Generator vs Discriminator
#   - Why GANs work (Nash equilibrium intuition for professionals)
#   - Build and train an MLP-based GAN on full MNIST (60K images)
#   - Diagnose training dynamics: when is D "winning" vs healthy balance
#   - Visualise generated digits, training progression, and loss dynamics
#   - Apply synthetic data generation for a Singapore insurance company
#     facing data scarcity under PDPA privacy regulations
#
# PREREQUISITES: M5/ex_1 (autoencoders — generative model foundations)
# ESTIMATED TIME: ~45 min
# DATASET: MNIST — 60,000 real 28x28 grayscale digits
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt



## PHASE 1 — THEORY: The Minimax Game


In [ ]:
# Imagine a counterfeiter (Generator) trying to produce fake banknotes
# that fool a detective (Discriminator). The counterfeiter never sees
# real banknotes — they only learn from the detective's feedback:
# "this one looks fake because the watermark is wrong."
#
# Over time:
#   - The counterfeiter improves their forgeries
#   - The detective gets better at spotting fakes
#   - Eventually they reach a standoff (Nash equilibrium) where the
#     detective can't tell real from fake — that's a trained GAN.
#
# Mathematically, this is a minimax game:
#   min_G max_D [ E[log D(x)] + E[log(1 - D(G(z)))] ]
#
# D wants to maximise: score real images high, fake images low
# G wants to minimise: make D score fake images high
#
# When D is perfectly confused (outputs 0.5 for everything),
# D_loss converges to ln(4) ≈ 1.386.
#
# MODE COLLAPSE: The biggest GAN failure mode. The counterfeiter finds
# ONE type of banknote that fools the detective and keeps making only
# that one. In MNIST terms: the generator only produces 1s because
# they're the simplest digit. We'll measure this with mode coverage.



GAN = Generator + Discriminator in a zero-sum game.

  Generator (counterfeiter):
    - Input: random noise z ~ N(0, 1)
    - Output: fake image that should look real
    - Goal: fool the discriminator

  Discriminator (detective):
    - Input: real OR fake image
    - Output: probability that the image is real
    - Goal: correctly classify real vs fake

  Training alternates:
    1. Train D on a batch of real + fake images
    2. Train G to make D output "real" for fake images

  Nash equilibrium: D outputs 0.5 for everything (can't tell the difference)

  KEY RISK — Mode collapse:
    G discovers one "easy" output (e.g., only digit 1) and stops exploring.
    We detect this by checking whether all 10 digit classes are generated.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: The Adversarial Minimax Game")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: Generator + Discriminator


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 2 — BUILD: Generator and Discriminator Networks")
print("=" * 70)

device = init_environment()
X_real, y_real, real_loader = load_mnist(device)
conn, tracker, exp_name, registry = setup_engines()

# Verify architectures
G_test = Generator().to(device)
D_test = Discriminator().to(device)
z_test = torch.randn(4, LATENT_DIM, device=device)

print(f"\n  Generator: {sum(p.numel() for p in G_test.parameters()):,} parameters")
print(f"    Input:  z ~ N(0, 1), dim={LATENT_DIM}")
print(f"    Output: {tuple(G_test(z_test).shape)} image (28x28)")
print(f"\n  Discriminator: {sum(p.numel() for p in D_test.parameters()):,} parameters")
print(f"    Input:  28x28 image")
print(f"    Output: {tuple(D_test(G_test(z_test)).shape)} scalar logit")



### Checkpoint 1


In [ ]:
assert G_test(z_test).shape == (4, 1, 28, 28), "Generator output shape wrong"
assert D_test(G_test(z_test)).shape == (4, 1), "Discriminator output shape wrong"
del G_test, D_test, z_test
print("\n--- Checkpoint 1 passed --- G and D architectures verified\n")



## PHASE 3 — TRAIN: Vanilla GAN with Binary Cross-Entropy


Train a vanilla GAN with BCE loss, logging to ExperimentTracker.


In [ ]:
# L_D = -E[log D(x)] - E[log(1 - D(G(z)))]    (discriminator loss)
# L_G = -E[log D(G(z))]                         (generator loss — non-saturating)
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: Vanilla GAN (BCEWithLogitsLoss)")
print("=" * 70)

EPOCHS = 15
LR = 2e-4


async def train_vanilla_gan(epochs: int = EPOCHS, lr: float = LR):
    G = Generator().to(device)
    D = Discriminator().to(device)
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    # TODO: Define BCE loss for adversarial training
    # Hint: bce = nn.BCEWithLogitsLoss()
    bce = ____
    g_losses, d_losses = [], []
    epoch_snapshots = {}

    # Capture initial state (random noise output)
    epoch_snapshots[0] = copy.deepcopy(G.state_dict())

    async with tracker.track(experiment=exp_name, run_name="vanilla_gan") as run:
        await run.log_params(
            {
                "architecture": "Vanilla_GAN_MLP",
                "latent_dim": str(LATENT_DIM),
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": "128",
                "loss_type": "BCEWithLogitsLoss",
                "optimizer": "Adam(0.5,0.999)",
            }
        )

        for epoch in range(epochs):
            eg, ed = [], []
            for (real_batch,) in real_loader:
                bs = real_batch.size(0)

                # ── Train Discriminator ──────────────────────────────
                # D sees real images (label=1) and fake images (label=0)
                z = torch.randn(bs, LATENT_DIM, device=device)
                fake = G(z).detach()
                # TODO: D loss = BCE on real (target=1) + BCE on fake (target=0)
                # Hint: loss_d = bce(D(real_batch), torch.ones(bs, 1, device=device))
                #              + bce(D(fake), torch.zeros(bs, 1, device=device))
                loss_d = ____
                opt_d.zero_grad()
                loss_d.backward()
                opt_d.step()

                # ── Train Generator ──────────────────────────────────
                # G wants D to classify its fakes as real (target=1)
                z = torch.randn(bs, LATENT_DIM, device=device)
                # TODO: G loss = BCE on D(G(z)) with target=1 (fool the discriminator)
                # Hint: loss_g = bce(D(G(z)), torch.ones(bs, 1, device=device))
                loss_g = ____
                opt_g.zero_grad()
                loss_g.backward()
                opt_g.step()

                eg.append(loss_g.item())
                ed.append(loss_d.item())

            avg_g, avg_d = float(np.mean(eg)), float(np.mean(ed))
            g_losses.append(avg_g)
            d_losses.append(avg_d)
            await run.log_metrics({"g_loss": avg_g, "d_loss": avg_d}, step=epoch + 1)
            print(
                f"  [Vanilla GAN] epoch {epoch+1:2d}/{epochs}  "
                f"D={avg_d:.3f}  G={avg_g:.3f}"
            )

            # Capture snapshots for progression visualisation
            if (epoch + 1) in {1, 5, 10, 15}:
                epoch_snapshots[epoch + 1] = copy.deepcopy(G.state_dict())

        await run.log_metrics(
            {"final_g_loss": g_losses[-1], "final_d_loss": d_losses[-1]}
        )

    return G, g_losses, d_losses, epoch_snapshots


print("\n  Training vanilla GAN on full MNIST (60K images)...")
G_gan, gan_g_losses, gan_d_losses, gan_snapshots  = await train_vanilla_gan()



### Checkpoint 2


In [ ]:
assert len(gan_g_losses) == EPOCHS, f"Expected {EPOCHS} epochs, got {len(gan_g_losses)}"
# INTERPRETATION: In a healthy GAN, D loss hovers around ln(4) ~ 1.386,
# meaning D is about 50% accurate (can't tell real from fake). If D loss
# drops to 0, D has "won" — it perfectly classifies everything — and G
# gets no useful gradient signal (training collapses).
print("\n--- Checkpoint 2 passed --- vanilla GAN trained\n")



## PHASE 4 — VISUALISE: Generated Gallery, Loss Curves, Progression


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: What Has the Generator Learned?")
print("=" * 70)

# 4A: Generated image gallery (8x8 grid)
print("\n  4A: Gallery of 64 generated digits")
G_gan.eval()
with torch.no_grad():
    z_gallery = torch.randn(64, LATENT_DIM, device=device)
    gallery_images = G_gan(z_gallery)

fig_gallery = plot_image_grid(
    gallery_images,
    title="Vanilla GAN — Generated Digits (8x8 Grid)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png"),
)
plt.show()

# 4B: Training progression (epoch 0 → 1 → 5 → 10 → 15)
print("\n  4B: Training progression — from noise to digits")
G_progression = Generator().to(device)
fig_progression = plot_training_progression(
    G_progression,
    device,
    gan_snapshots,
    title="Vanilla GAN — Training Progression (Epoch 0 → 15)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png"),
)
plt.show()
# Reload the final state after progression visualisation
G_gan.load_state_dict(gan_snapshots[max(gan_snapshots.keys())])
G_gan.eval()

# 4C: G vs D loss dynamics
print("\n  4C: Generator vs Discriminator loss curves")
fig_losses = plot_loss_curves(
    gan_g_losses,
    gan_d_losses,
    title="Vanilla GAN — Training Dynamics",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_losses.png"),
)
plt.show()

# 4D: Latent space interpolation
print("\n  4D: Latent space interpolation — smooth transitions between digits")
fig_interp = plot_latent_interpolation(
    G_gan,
    device,
    title="Vanilla GAN — Latent Interpolation",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_interpolation.png"),
)
plt.show()



### Checkpoint 3


In [ ]:
import os

assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png")
), "Gallery image should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png")
), "Progression image should exist"
# INTERPRETATION: The gallery shows whether the generator produces
# diverse, recognisable digits. The progression shows HOW it learns:
# epoch 0 is pure noise, early epochs show blurry shapes, later epochs
# show distinct digits. If all generated images look the same, that's
# mode collapse — the generator found one "easy" output.
print("\n--- Checkpoint 3 passed --- vanilla GAN visualisations complete\n")

# Also save training curves with ModelVisualizer (HTML interactive)
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()
fig_html = viz.training_history(
    metrics={"GAN G loss": gan_g_losses, "GAN D loss": gan_d_losses},
    x_label="Epoch",
    y_label="Loss",
)
fig_html.write_html(str(OUTPUT_DIR / "ex_5_01_vanilla_training.html"))
print("  Interactive training curves saved to ex_5_01_vanilla_training.html")



## PHASE 5 — APPLY: Synthetic Data for Singapore Insurance Under PDPA


BUSINESS SCENARIO:
  You are a data scientist at a Singapore life insurance company.
  Your team needs to build a fraud detection model, but the Personal
  Data Protection Act (PDPA) restricts how you can use real policyholder
  records for model training. You have only 2,000 real claim records
  (too few for robust ML), and PDPA compliance review takes 6 months
  for new data access requests.

  SOLUTION: Train a GAN on the limited real data to generate synthetic
  policyholder profiles that preserve statistical properties without
  containing any real person's data. The synthetic data supplements
  real data for model training — no PDPA issues because no real
  personal data is used.

  BUSINESS IMPACT:
  - Model training data increased from 2,000 to 20,000+ records
  - No PDPA compliance delay (synthetic data is not personal data)
  - Fraud detection recall improves from 62% to 78%
  - Estimated annual fraud savings: S$4.2M additional recovered claims


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Data for Insurance (PDPA Compliance)")
print("=" * 70)
print(
)

# Simulate the insurance scenario using MNIST as a proxy:
# Real policyholder "profiles" = real MNIST digits (limited sample)
# Synthetic profiles = GAN-generated digits
# We compare the statistical distributions to validate quality.

print("\n  Simulating the insurance data scenario with MNIST as proxy...")

# Step 1: Take a small "real" sample (simulating limited insurance data)
rng = np.random.default_rng(42)
small_sample_idx = rng.choice(len(X_real), 2000, replace=False)
X_small_real = X_real[small_sample_idx]
y_small_real = y_real[small_sample_idx]

print(f"  'Real' policyholder records: {len(X_small_real)}")

# Step 2: Generate synthetic data to supplement
# TODO: Generate 18,000 synthetic records using the trained generator
# Hint: Set G_gan to eval mode, generate z from N(0,1) with shape (18000, LATENT_DIM),
#       pass through G_gan with torch.no_grad()
G_gan.eval()
with torch.no_grad():
    z_synthetic = ____  # TODO: torch.randn(18000, LATENT_DIM, device=device)
    X_synthetic = ____  # TODO: G_gan(z_synthetic)

print(f"  Synthetic records generated: {len(X_synthetic)}")
print(f"  Combined dataset: {len(X_small_real) + len(X_synthetic)} records")

# Step 3: Compare real vs synthetic pixel distributions
# TODO: Flatten both tensors for distribution comparison
# Hint: X_real_flat = X_small_real.view(-1).cpu().numpy()
#       X_synth_flat = X_synthetic.view(-1).cpu().numpy()
X_real_flat = ____
X_synth_flat = ____

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "Real vs Synthetic Distribution Comparison\n"
    "(Insurance Policyholder Profile Proxy)",
    fontsize=14,
    fontweight="bold",
)

# TODO: Plot overlapping histograms for feature distribution
# Hint: axes[0].hist(X_real_flat, bins=50, alpha=0.6, label="Real (2K records)", density=True)
#       axes[0].hist(X_synth_flat, bins=50, alpha=0.6, label="Synthetic (18K)", density=True)
____
____
axes[0].set_xlabel("Feature Value", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("Feature Distribution Overlap", fontsize=13)
axes[0].legend(fontsize=11)

# TODO: Plot mean feature values per sample
# Hint: real_means = X_small_real.view(len(X_small_real), -1).mean(dim=1).cpu().numpy()
#       synth_means = X_synthetic.view(len(X_synthetic), -1).mean(dim=1).cpu().numpy()
real_means = ____
synth_means = ____
axes[1].hist(real_means, bins=30, alpha=0.6, label="Real", density=True)
axes[1].hist(synth_means, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[1].set_xlabel("Mean Feature Value per Record", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("Record-Level Distribution", fontsize=13)
axes[1].legend(fontsize=11)

# TODO: Plot variance per sample
# Hint: real_vars = X_small_real.view(len(X_small_real), -1).var(dim=1).cpu().numpy()
#       synth_vars = X_synthetic.view(len(X_synthetic), -1).var(dim=1).cpu().numpy()
real_vars = ____
synth_vars = ____
axes[2].hist(real_vars, bins=30, alpha=0.6, label="Real", density=True)
axes[2].hist(synth_vars, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[2].set_xlabel("Feature Variance per Record", fontsize=12)
axes[2].set_ylabel("Density", fontsize=12)
axes[2].set_title("Record Diversity", fontsize=13)
axes[2].legend(fontsize=11)

plt.tight_layout()
fig.savefig(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Distribution comparison saved")

# Step 4: Stakeholder-ready summary
print("\n  ┌────────────────────────────────────────────────────────────┐")
print("  │  STAKEHOLDER SUMMARY: Synthetic Data Quality Assessment   │")
print("  ├────────────────────────────────────────────────────────────┤")
print(f"  │  Real records available:       {len(X_small_real):>8,}                  │")
print(f"  │  Synthetic records generated:  {len(X_synthetic):>8,}                  │")
print(
    f"  │  Combined training set:        {len(X_small_real)+len(X_synthetic):>8,}                  │"
)
print(
    f"  │  Data augmentation factor:     {(len(X_small_real)+len(X_synthetic))/len(X_small_real):.1f}x                       │"
)
print("  │                                                            │")
real_mean_val = float(np.mean(real_means))
synth_mean_val = float(np.mean(synth_means))
mean_diff_pct = abs(real_mean_val - synth_mean_val) / (abs(real_mean_val) + 1e-8) * 100
print(f"  │  Mean feature difference:      {mean_diff_pct:>7.1f}%                  │")
print("  │  PDPA compliance:              No personal data used      │")
print("  │  Status:                       READY FOR MODEL TRAINING   │")
print("  └────────────────────────────────────────────────────────────┘")



### Checkpoint 4


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png")
), "Distribution comparison should exist"
assert len(X_synthetic) == 18000, "Should generate 18K synthetic records"
print("\n--- Checkpoint 4 passed --- insurance application complete\n")



## Cleanup


In [ ]:
await close_engines(conn)



## REFLECTION


VANILLA GAN FUNDAMENTALS:
  [x] The minimax game: Generator (counterfeiter) vs Discriminator (detective)
  [x] Binary cross-entropy loss for adversarial training
  [x] Nash equilibrium: D loss converging to ln(4) ~ 1.386
  [x] Mode collapse risk: generator producing only "easy" outputs

  VISUAL INTUITION:
  [x] Generated digit gallery (8x8 grid) — can you read the digits?
  [x] Training progression: noise → blurry shapes → recognisable digits
  [x] G vs D loss curves — healthy balance vs D domination
  [x] Latent interpolation — smooth transitions prove learned manifold

  REAL-WORLD APPLICATION:
  [x] Synthetic data generation for PDPA-compliant model training
  [x] Statistical validation: real vs synthetic distribution comparison
  [x] Business impact quantification: S$4.2M in additional fraud recovery

  KEY INSIGHT:
  Vanilla GANs work but are UNSTABLE. The BCE loss gives zero gradient
  when D perfectly separates real from fake (JS divergence saturates).
  This means training can suddenly collapse with no warning.

  Next: Exercise 5.2 — WGAN-GP solves this instability with
  Wasserstein distance (smooth gradients even when distributions
  don't overlap) and gradient penalty (replaces weight clipping).


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — Vanilla GAN (track G + D separately)


In [ ]:
# GANs have TWO networks training in an adversarial loop. We run the
# Prescription Pad on BOTH the Generator and the Discriminator so we
# can see which side is "winning" and which side is starving for
# signal. The `train_losses` we replay into each diag are the per-
# epoch losses already captured above.
from kailash_ml.diagnostics import run_diagnostic_checkpoint
import torch.nn.functional as _F


def _g_loss(m, batch):
    # Re-run the G objective: BCE against "real" label.
    bs = batch[0].size(0)
    z = torch.randn(bs, LATENT_DIM, device=device)
    fake = m(z)
    # Need a D to score fakes — use G_gan's paired D is unavailable here,
    # so we build a tiny surrogate scoring head for the checkpoint pass.
    return _F.mse_loss(fake, batch[0])  # proxy reconstruction signal


def _d_loss(m, batch):
    # Re-run the D objective on real + fake for a readable gradient view.
    bs = batch[0].size(0)
    real_score = m(batch[0])
    z = torch.randn(bs, LATENT_DIM, device=device)
    fake_score = m(G_gan(z).detach())
    return _F.binary_cross_entropy_with_logits(
        real_score, torch.ones_like(real_score)
    ) + _F.binary_cross_entropy_with_logits(fake_score, torch.zeros_like(fake_score))


print("\n── Diagnostic Report (Generator) ──")
g_diag, g_findings = run_diagnostic_checkpoint(
    G_gan,
    real_loader,
    _g_loss,
    title="Vanilla GAN — Generator",
    n_batches=6,
    train_losses=gan_g_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (reference pattern — vanilla GAN on MNIST) ═



## DL Diagnostics Report — Prescription Pad (Generator)


In [ ]:
#   [!] Gradient flow (WARNING): Generator gradients become SMALL
#       when D wins (D_loss approaches 0). RMS on early G layers
#       drops below 1e-5 — the BCE objective saturates and the
#       Generator stops learning. Classic "D dominance" failure.
#   [!] Activations    (WARNING): Generator tanh outputs may
#       cluster near a single mode (e.g., all 1s) — this is the
#       visible signature of MODE COLLAPSE on MNIST. Check the
#       gallery: if 50+ of 64 tiles are the same digit, confirmed.
#   [~] Loss trend     (MIXED): G loss and D loss oscillate —
#       typical adversarial dynamics. D loss near ln(4)≈1.386 is
#       healthy; D loss near 0 means D won and G is starving.



## STUDENT INTERPRETATION GUIDE — reading the GAN Prescription Pad:

 [BLOOD TEST] Generator gradient RMS tracks the adversarial
    balance. When D "wins" (D_loss -> 0), the saturating BCE
    gradient in G collapses to ~0 and G stops updating. This
    is THE reason slide 5.5 (GANs) motivates WGAN-GP — the
    Wasserstein loss provides a gradient EVERYWHERE, even when
    D perfectly separates the distributions.
    >> Prescription Pad: if G's Blood Test is WARNING and D's
       loss is near 0, switch to WGAN-GP (ex_5/02) OR add label
       smoothing (real_labels = 0.9 instead of 1.0) OR reduce
       D's learning rate relative to G's.

 [X-RAY] Mode collapse — reference the Prescription Pad row:
    "mode collapse → diversify noise, add minibatch
    discrimination, use WGAN-GP". The X-Ray detects this as
    collapsed activation diversity in the Generator's final
    conv/linear layer. Slide 5.5 illustrates this with the
    "only 1s" failure mode. Count distinct digits in your
    gallery — healthy GAN produces all 10 classes, collapsed
    GAN produces 1-3.
    >> Prescription Pad: minibatch discrimination OR feature
       matching OR WGAN-GP (Wasserstein doesn't suffer from
       this as severely as BCE).

 [STETHOSCOPE] GAN loss curves are NOT the usual "monotonically
    down" shape. Healthy GAN shows OSCILLATING losses — G and D
    trade wins as they co-evolve. Flat-lining D loss near 0 is
    the failure signature (D has won permanently). Flat-lining
    D loss near ln(4)≈1.386 is the Nash equilibrium (ideal).
    >> Prescription Pad: if D loss flat-lines at 0, halt training
       and either reduce D capacity OR increase G capacity OR
       switch to WGAN-GP.

 FIVE-INSTRUMENT TAKEAWAY: GANs are the one architecture where
 HEALTHY loss curves look UNHEALTHY by supervised-learning
 standards. The Prescription Pad's value is translation — it
 reads the oscillations as signal, not noise. Slide 5.5 uses
 these reports to motivate WGAN-GP in the next file: every
 WARNING above becomes HEALTHY there.

 CONNECT TO SLIDE 5.5 (GANs): slide claims "vanilla GAN is
 unstable; WGAN-GP fixes the gradient-signal problem". The
 G-side WARNING above + ex_5/02's all-HEALTHY report is the
 empirical proof of that claim.


### Checkpoint 2


In [ ]:
assert len(gan_g_losses) == EPOCHS, f"Expected {EPOCHS} epochs, got {len(gan_g_losses)}"
# INTERPRETATION: In a healthy GAN, D loss hovers around ln(4) ~ 1.386,
# meaning D is about 50% accurate (can't tell real from fake). If D loss
# drops to 0, D has "won" — it perfectly classifies everything — and G
# gets no useful gradient signal (training collapses).
print("\n--- Checkpoint 2 passed --- vanilla GAN trained\n")



## PHASE 4 — VISUALISE: Generated Gallery, Loss Curves, Progression


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: What Has the Generator Learned?")
print("=" * 70)

# 4A: Generated image gallery (8x8 grid)
print("\n  4A: Gallery of 64 generated digits")
G_gan.eval()
with torch.no_grad():
    z_gallery = torch.randn(64, LATENT_DIM, device=device)
    gallery_images = G_gan(z_gallery)

fig_gallery = plot_image_grid(
    gallery_images,
    title="Vanilla GAN — Generated Digits (8x8 Grid)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png"),
)
plt.show()

# 4B: Training progression (epoch 0 → 1 → 5 → 10 → 15)
print("\n  4B: Training progression — from noise to digits")
G_progression = Generator().to(device)
fig_progression = plot_training_progression(
    G_progression,
    device,
    gan_snapshots,
    title="Vanilla GAN — Training Progression (Epoch 0 → 15)",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png"),
)
plt.show()
# Reload the final state after progression visualisation
G_gan.load_state_dict(gan_snapshots[max(gan_snapshots.keys())])
G_gan.eval()

# 4C: G vs D loss dynamics
print("\n  4C: Generator vs Discriminator loss curves")
fig_losses = plot_loss_curves(
    gan_g_losses,
    gan_d_losses,
    title="Vanilla GAN — Training Dynamics",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_losses.png"),
)
plt.show()

# 4D: Latent space interpolation
print("\n  4D: Latent space interpolation — smooth transitions between digits")
fig_interp = plot_latent_interpolation(
    G_gan,
    device,
    title="Vanilla GAN — Latent Interpolation",
    save_path=str(OUTPUT_DIR / "ex_5_01_vanilla_interpolation.png"),
)
plt.show()



### Checkpoint 3


In [ ]:
import os

assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_gallery.png")
), "Gallery image should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_progression.png")
), "Progression image should exist"
# INTERPRETATION: The gallery shows whether the generator produces
# diverse, recognisable digits. The progression shows HOW it learns:
# epoch 0 is pure noise, early epochs show blurry shapes, later epochs
# show distinct digits. If all generated images look the same, that's
# mode collapse — the generator found one "easy" output.
print("\n--- Checkpoint 3 passed --- vanilla GAN visualisations complete\n")

# Also save training curves with ModelVisualizer (HTML interactive)
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()
fig_html = viz.training_history(
    metrics={"GAN G loss": gan_g_losses, "GAN D loss": gan_d_losses},
    x_label="Epoch",
    y_label="Loss",
)
fig_html.write_html(str(OUTPUT_DIR / "ex_5_01_vanilla_training.html"))
print("  Interactive training curves saved to ex_5_01_vanilla_training.html")



## PHASE 5 — APPLY: Synthetic Data for Singapore Insurance Under PDPA


BUSINESS SCENARIO:
  You are a data scientist at a Singapore life insurance company.
  Your team needs to build a fraud detection model, but the Personal
  Data Protection Act (PDPA) restricts how you can use real policyholder
  records for model training. You have only 2,000 real claim records
  (too few for robust ML), and PDPA compliance review takes 6 months
  for new data access requests.

  SOLUTION: Train a GAN on the limited real data to generate synthetic
  policyholder profiles that preserve statistical properties without
  containing any real person's data. The synthetic data supplements
  real data for model training — no PDPA issues because no real
  personal data is used.

  BUSINESS IMPACT:
  - Model training data increased from 2,000 to 20,000+ records
  - No PDPA compliance delay (synthetic data is not personal data)
  - Fraud detection recall improves from 62% to 78%
  - Estimated annual fraud savings: S$4.2M additional recovered claims


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Data for Insurance (PDPA Compliance)")
print("=" * 70)
print(
)

# Simulate the insurance scenario using MNIST as a proxy:
# Real policyholder "profiles" = real MNIST digits (limited sample)
# Synthetic profiles = GAN-generated digits
# We compare the statistical distributions to validate quality.

print("\n  Simulating the insurance data scenario with MNIST as proxy...")

# Step 1: Take a small "real" sample (simulating limited insurance data)
rng = np.random.default_rng(42)
small_sample_idx = rng.choice(len(X_real), 2000, replace=False)
X_small_real = X_real[small_sample_idx]
y_small_real = y_real[small_sample_idx]

print(f"  'Real' policyholder records: {len(X_small_real)}")

# Step 2: Generate synthetic data to supplement
G_gan.eval()
with torch.no_grad():
    z_synthetic = torch.randn(18000, LATENT_DIM, device=device)
    X_synthetic = G_gan(z_synthetic)

print(f"  Synthetic records generated: {len(X_synthetic)}")
print(f"  Combined dataset: {len(X_small_real) + len(X_synthetic)} records")

# Step 3: Compare real vs synthetic pixel distributions
X_real_flat = X_small_real.view(-1).cpu().numpy()
X_synth_flat = X_synthetic.view(-1).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "Real vs Synthetic Distribution Comparison\n"
    "(Insurance Policyholder Profile Proxy)",
    fontsize=14,
    fontweight="bold",
)

# Pixel intensity distribution
axes[0].hist(X_real_flat, bins=50, alpha=0.6, label="Real (2K records)", density=True)
axes[0].hist(X_synth_flat, bins=50, alpha=0.6, label="Synthetic (18K)", density=True)
axes[0].set_xlabel("Feature Value", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("Feature Distribution Overlap", fontsize=13)
axes[0].legend(fontsize=11)

# Mean feature values per sample
real_means = X_small_real.view(len(X_small_real), -1).mean(dim=1).cpu().numpy()
synth_means = X_synthetic.view(len(X_synthetic), -1).mean(dim=1).cpu().numpy()
axes[1].hist(real_means, bins=30, alpha=0.6, label="Real", density=True)
axes[1].hist(synth_means, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[1].set_xlabel("Mean Feature Value per Record", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("Record-Level Distribution", fontsize=13)
axes[1].legend(fontsize=11)

# Variance per sample
real_vars = X_small_real.view(len(X_small_real), -1).var(dim=1).cpu().numpy()
synth_vars = X_synthetic.view(len(X_synthetic), -1).var(dim=1).cpu().numpy()
axes[2].hist(real_vars, bins=30, alpha=0.6, label="Real", density=True)
axes[2].hist(synth_vars, bins=30, alpha=0.6, label="Synthetic", density=True)
axes[2].set_xlabel("Feature Variance per Record", fontsize=12)
axes[2].set_ylabel("Density", fontsize=12)
axes[2].set_title("Record Diversity", fontsize=13)
axes[2].legend(fontsize=11)

plt.tight_layout()
fig.savefig(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()
print("  Distribution comparison saved")

# Step 4: Stakeholder-ready summary
print("\n  ┌────────────────────────────────────────────────────────────┐")
print("  │  STAKEHOLDER SUMMARY: Synthetic Data Quality Assessment   │")
print("  ├────────────────────────────────────────────────────────────┤")
print(f"  │  Real records available:       {len(X_small_real):>8,}                  │")
print(f"  │  Synthetic records generated:  {len(X_synthetic):>8,}                  │")
print(
    f"  │  Combined training set:        {len(X_small_real)+len(X_synthetic):>8,}                  │"
)
print(
    f"  │  Data augmentation factor:     {(len(X_small_real)+len(X_synthetic))/len(X_small_real):.1f}x                       │"
)
print("  │                                                            │")
real_mean_val = float(np.mean(real_means))
synth_mean_val = float(np.mean(synth_means))
mean_diff_pct = abs(real_mean_val - synth_mean_val) / (abs(real_mean_val) + 1e-8) * 100
print(f"  │  Mean feature difference:      {mean_diff_pct:>7.1f}%                  │")
print("  │  PDPA compliance:              No personal data used      │")
print("  │  Status:                       READY FOR MODEL TRAINING   │")
print("  └────────────────────────────────────────────────────────────┘")



### Checkpoint 4


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_01_vanilla_real_vs_synthetic.png")
), "Distribution comparison should exist"
assert len(X_synthetic) == 18000, "Should generate 18K synthetic records"
print("\n--- Checkpoint 4 passed --- insurance application complete\n")



## Cleanup


In [ ]:
await close_engines(conn)



## REFLECTION


VANILLA GAN FUNDAMENTALS:
  [x] The minimax game: Generator (counterfeiter) vs Discriminator (detective)
  [x] Binary cross-entropy loss for adversarial training
  [x] Nash equilibrium: D loss converging to ln(4) ~ 1.386
  [x] Mode collapse risk: generator producing only "easy" outputs

  VISUAL INTUITION:
  [x] Generated digit gallery (8x8 grid) — can you read the digits?
  [x] Training progression: noise → blurry shapes → recognisable digits
  [x] G vs D loss curves — healthy balance vs D domination
  [x] Latent interpolation — smooth transitions prove learned manifold

  REAL-WORLD APPLICATION:
  [x] Synthetic data generation for PDPA-compliant model training
  [x] Statistical validation: real vs synthetic distribution comparison
  [x] Business impact quantification: S$4.2M in additional fraud recovery

  KEY INSIGHT:
  Vanilla GANs work but are UNSTABLE. The BCE loss gives zero gradient
  when D perfectly separates real from fake (JS divergence saturates).
  This means training can suddenly collapse with no warning.

  Next: Exercise 5.2 — WGAN-GP solves this instability with
  Wasserstein distance (smooth gradients even when distributions
  don't overlap) and gradient penalty (replaces weight clipping).


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

